In [ ]:
import os, shutil, subprocess,sys
import torch
import json
import numpy as np

PDFTEXBIN = '/opt/local/texlive/texlive-2018/2018/bin/x86_64-linux/pdflatex'
TEXBIN = os.path.dirname(PDFTEXBIN)

os.environ['PATH'] = TEXBIN + os.pathsep + os.environ.get('PATH', '')


import matplotlib as mpl

def _get_state(ckpt_path: str):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict):
        # common patterns: {"state_dict": ...}, {"model": ...}, {"unet_p": ...}, etc.
        for k in ("unet", "q"):
            if k in ckpt and isinstance(ckpt[k], dict):
                return ckpt[k]
        # sometimes the ckpt itself is a state_dict
        if all(isinstance(v, torch.Tensor) for v in ckpt.values()):
            return ckpt
    raise ValueError(f"Cannot find state_dict in {ckpt_path}. Keys: {list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt)}")

@torch.no_grad()
def rel_l2_param_delta(ckpt1: str, ckpt2: str) -> dict:
    s1 = _get_state(ckpt1)
    s2 = _get_state(ckpt2)

    num = 0.0
    den = 0.0
    missing = 0

    for k, v1 in s1.items():
        v2 = s2.get(k, None)
        if v2 is None:
            missing += 1
            continue
        if not (torch.is_tensor(v1) and torch.is_tensor(v2)):
            continue
        if v1.shape != v2.shape:
            continue

        d = (v2.float() - v1.float()).view(-1)
        num += float(torch.dot(d, d))
        den += float(torch.dot(v1.float().view(-1), v1.float().view(-1)))

    num = num ** 0.5
    den = den ** 0.5
    return {
        "abs_l2_delta": num,
        "l2_norm_ref": den,
        "rel_l2_delta": (num / (den + 1e-12)),
        "missing_keys_in_ckpt2": missing,
    }

def read_jsonl_key_to_list(path: str, key: str):
    vals = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                d = json.loads(line)
            except json.JSONDecodeError:
                continue
            if not isinstance(d, dict):
                continue
            v = d.get(key, None)
            vals.append(v)
    last_val = vals[-1]
    last_5_val_ave = sum(vals[-5:]) / 5
    last_3_val_ave = sum(vals[-3:]) / 3
    return vals, last_val, last_5_val_ave, last_3_val_ave

mpl.use("pgf")
mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "font.family": "serif",
    "pgf.preamble": r"""
\usepackage[T1]{fontenc}
\usepackage{newtxtext,newtxmath}
""",
    "font.size": 8,          
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "lines.linewidth": 2.0,
    "pdf.fonttype": 42,     
    "ps.fonttype": 42,
})

import matplotlib.pyplot as plt
setting = 4  # A4
cu5or6 = 6  # curation ratio 0.5 or 0.6

iteraA = [0, 5, 10, 15, 20, 25, 30, 35, 39, 40, 42, 44, 46, 48, 50, 52, 54]  # focus on these iterations for better visualization

crsSet = [[0, 1], [2, 8], [3, 4], [7, 3], [2, 3], [7, 8]][setting-1]
fileName = f"cifar_sam8k_r5sy0cu{cu5or6}_pcrs{crsSet[0]}_qcrs{crsSet[1]}_cIter4" if cu5or6 == 5 else f"cifar_psam10k_r4cu6_qsam8k_r5cu5_pcrs{crsSet[0]}_qcrs{crsSet[1]}_cIter4"
p_deltaNorms = []
p_norms = []
q_deltaNorms = []
q_norms = []

p_rewards = read_jsonl_key_to_list(f"runs/{fileName}/log.jsonl", "p_reward_on_pgen(p reward)")[0]
p_rewards = [p_rewards[x] for x in iteraA]
q_rewards = read_jsonl_key_to_list(f"runs/{fileName}/log.jsonl", "q_reward_on_qgen(q reward)")[0]
q_rewards = [q_rewards[x] for x in iteraA]

for idx in range(len(iteraA)-1):
    ckpt_p_later = f"runs/{fileName}/ckpt/p_unet_iter_{iteraA[idx+1]:03d}.pt"
    ckpt_p_before = f"runs/{fileName}/ckpt/p_unet_iter_{iteraA[idx]:03d}.pt"
    ckpt_q_later = f"runs/{fileName}/ckpt/q_unet_iter_{iteraA[idx+1]:03d}.pt"
    ckpt_q_before = f"runs/{fileName}/ckpt/q_unet_iter_{iteraA[idx]:03d}.pt"

    out = rel_l2_param_delta(ckpt_p_before, ckpt_p_later)
    p_deltaNorms.append(out["rel_l2_delta"])
    p_norms.append(out["l2_norm_ref"])
    out = rel_l2_param_delta(ckpt_q_before, ckpt_q_later)
    q_deltaNorms.append(out["rel_l2_delta"])
    q_norms.append(out["l2_norm_ref"])

fileName_A1_cu5 = "cifar_sam8k_r5sy0cu5_pcrs0_qcrs1_cIter4"
fileName_A1_cu6 = "cifar_psam10k_r4cu6_qsam8k_r5cu5_pcrs0_qcrs1_cIter4"

fileName_A2_cu5 = "cifar_sam8k_r5sy0cu5_pcrs2_qcrs8_cIter4"
fileName_A2_cu6 = "cifar_psam10k_r4cu6_qsam8k_r5cu5_pcrs2_qcrs8_cIter4"

fileName_A3_cu5 = "cifar_sam8k_r5sy0cu5_pcrs3_qcrs4_cIter4"
fileName_A3_cu6 = "cifar_psam10k_r4cu6_qsam8k_r5cu5_pcrs3_qcrs4_cIter4"

fileName_A4_cu5 = "cifar_sam8k_r5sy0cu5_pcrs7_qcrs3_cIter4"
fileName_A4_cu6 = "cifar_psam10k_r4cu6_qsam8k_r5cu5_pcrs7_qcrs3_cIter4"

fileName_A5_cu5 = "cifar_sam8k_r5sy0cu5_pcrs2_qcrs3_cIter4"
fileName_A5_cu6 = "cifar_psam10k_r4cu6_qsam8k_r5cu5_pcrs2_qcrs3_cIter4"

fileName_A6_cu5 = "cifar_sam8k_r5sy0cu5_pcrs7_qcrs8_cIter4"
fileName_A6_cu6 = "cifar_psam10k_r4cu6_qsam8k_r5cu5_pcrs7_qcrs8_cIter4"

p_rewards_A1_cu5 = read_jsonl_key_to_list(f"runs/{fileName_A1_cu5}/log.jsonl", "p_reward_on_pgen(p reward)")[0]
q_rewards_A1_cu5 = read_jsonl_key_to_list(f"runs/{fileName_A1_cu5}/log.jsonl", "q_reward_on_qgen(q reward)")[0]
p_rewards_A1_cu6 = read_jsonl_key_to_list(f"runs/{fileName_A1_cu6}/log.jsonl", "p_reward_on_pgen(p reward)")[0]
q_rewards_A1_cu6 = read_jsonl_key_to_list(f"runs/{fileName_A1_cu6}/log.jsonl", "q_reward_on_qgen(q reward)")[0]

p_rewards_A2_cu5 = read_jsonl_key_to_list(f"runs/{fileName_A2_cu5}/log.jsonl", "p_reward_on_pgen(p reward)")[0]
q_rewards_A2_cu5 = read_jsonl_key_to_list(f"runs/{fileName_A2_cu5}/log.jsonl", "q_reward_on_qgen(q reward)")[0]
p_rewards_A2_cu6 = read_jsonl_key_to_list(f"runs/{fileName_A2_cu6}/log.jsonl", "p_reward_on_pgen(p reward)")[0]
q_rewards_A2_cu6 = read_jsonl_key_to_list(f"runs/{fileName_A2_cu6}/log.jsonl", "q_reward_on_qgen(q reward)")[0]

p_rewards_A3_cu5 = read_jsonl_key_to_list(f"runs/{fileName_A3_cu5}/log.jsonl", "p_reward_on_pgen(p reward)")[0]
q_rewards_A3_cu5 = read_jsonl_key_to_list(f"runs/{fileName_A3_cu5}/log.jsonl", "q_reward_on_qgen(q reward)")[0]
p_rewards_A3_cu6 = read_jsonl_key_to_list(f"runs/{fileName_A3_cu6}/log.jsonl", "p_reward_on_pgen(p reward)")[0]
q_rewards_A3_cu6 = read_jsonl_key_to_list(f"runs/{fileName_A3_cu6}/log.jsonl", "q_reward_on_qgen(q reward)")[0]

p_rewards_A4_cu5 = read_jsonl_key_to_list(f"runs/{fileName_A4_cu5}/log.jsonl", "p_reward_on_pgen(p reward)")[0]
q_rewards_A4_cu5 = read_jsonl_key_to_list(f"runs/{fileName_A4_cu5}/log.jsonl", "q_reward_on_qgen(q reward)")[0]
p_rewards_A4_cu6 = read_jsonl_key_to_list(f"runs/{fileName_A4_cu6}/log.jsonl", "p_reward_on_pgen(p reward)")[0]
q_rewards_A4_cu6 = read_jsonl_key_to_list(f"runs/{fileName_A4_cu6}/log.jsonl", "q_reward_on_qgen(q reward)")[0]

p_rewards_A5_cu5 = read_jsonl_key_to_list(f"runs/{fileName_A5_cu5}/log.jsonl", "p_reward_on_pgen(p reward)")[0]
q_rewards_A5_cu5 = read_jsonl_key_to_list(f"runs/{fileName_A5_cu5}/log.jsonl", "q_reward_on_qgen(q reward)")[0]
p_rewards_A5_cu6 = read_jsonl_key_to_list(f"runs/{fileName_A5_cu6}/log.jsonl", "p_reward_on_pgen(p reward)")[0]
q_rewards_A5_cu6 = read_jsonl_key_to_list(f"runs/{fileName_A5_cu6}/log.jsonl", "q_reward_on_qgen(q reward)")[0]

p_rewards_A6_cu5 = read_jsonl_key_to_list(f"runs/{fileName_A6_cu5}/log.jsonl", "p_reward_on_pgen(p reward)")[0]
q_rewards_A6_cu5 = read_jsonl_key_to_list(f"runs/{fileName_A6_cu5}/log.jsonl", "q_reward_on_qgen(q reward)")[0]
p_rewards_A6_cu6 = read_jsonl_key_to_list(f"runs/{fileName_A6_cu6}/log.jsonl", "p_reward_on_pgen(p reward)")[0]
q_rewards_A6_cu6 = read_jsonl_key_to_list(f"runs/{fileName_A6_cu6}/log.jsonl", "q_reward_on_qgen(q reward)")[0]
                    

t_idx = -1
p_final_reward_A1_cu5 = p_rewards_A1_cu5[t_idx]
q_final_reward_A1_cu5 = q_rewards_A1_cu5[t_idx]
p_final_reward_A1_cu6 = p_rewards_A1_cu6[t_idx]
q_final_reward_A1_cu6 = q_rewards_A1_cu6[t_idx]

p_final_reward_A2_cu5 = p_rewards_A2_cu5[t_idx]    
q_final_reward_A2_cu5 = q_rewards_A2_cu5[t_idx]
p_final_reward_A2_cu6 = p_rewards_A2_cu6[t_idx]    
q_final_reward_A2_cu6 = q_rewards_A2_cu6[t_idx]

p_final_reward_A3_cu5 = p_rewards_A3_cu5[t_idx]    
q_final_reward_A3_cu5 = q_rewards_A3_cu5[t_idx]
p_final_reward_A3_cu6 = p_rewards_A3_cu6[t_idx]    
q_final_reward_A3_cu6 = q_rewards_A3_cu6[t_idx]

p_final_reward_A4_cu5 = p_rewards_A4_cu5[t_idx]    
q_final_reward_A4_cu5 = q_rewards_A4_cu5[t_idx]
p_final_reward_A4_cu6 = p_rewards_A4_cu6[t_idx]    
q_final_reward_A4_cu6 = q_rewards_A4_cu6[t_idx]

p_final_reward_A5_cu5 = p_rewards_A5_cu5[t_idx]
q_final_reward_A5_cu5 = q_rewards_A5_cu5[t_idx]
p_final_reward_A5_cu6 = p_rewards_A5_cu6[t_idx]
q_final_reward_A5_cu6 = q_rewards_A5_cu6[t_idx]

p_final_reward_A6_cu5 = p_rewards_A6_cu5[t_idx]
q_final_reward_A6_cu5 = q_rewards_A6_cu5[t_idx]
p_final_reward_A6_cu6 = p_rewards_A6_cu6[t_idx]
q_final_reward_A6_cu6 = q_rewards_A6_cu6[t_idx]

p_rewards_final = [
    p_final_reward_A1_cu5, p_final_reward_A1_cu6,
    p_final_reward_A2_cu5, p_final_reward_A2_cu6,
    p_final_reward_A3_cu5, p_final_reward_A3_cu6,
    p_final_reward_A4_cu5, p_final_reward_A4_cu6, 
    p_final_reward_A5_cu5, p_final_reward_A5_cu6,
    p_final_reward_A6_cu5, p_final_reward_A6_cu6,
]
q_rewards_final = [
    q_final_reward_A1_cu5, q_final_reward_A1_cu6,
    q_final_reward_A2_cu5, q_final_reward_A2_cu6,
    q_final_reward_A3_cu5, q_final_reward_A3_cu6,
    q_final_reward_A4_cu5, q_final_reward_A4_cu6, 
    q_final_reward_A5_cu5, q_final_reward_A5_cu6,
    q_final_reward_A6_cu5, q_final_reward_A6_cu6,
]


In [ ]:
TEXTWIDTH_IN = 6.75
# fig, axes = plt.subplots(1, 3, figsize=(TEXTWIDTH_IN, 2.4), gridspec_kw={'width_ratios': [1.8, 1, 1], 'wspace': 0.5})

fig = plt.figure(figsize=(TEXTWIDTH_IN, 1.75))
spacer_ratio=0.08
gs = fig.add_gridspec(1, 4, width_ratios=[1.8, spacer_ratio, 1, 1], wspace=0.25)
axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 2]),
    fig.add_subplot(gs[0, 3]),
]
ax_spacer = fig.add_subplot(gs[0, 1])
ax_spacer.axis("off")

markSize = 3
linewidth = 1.5
labelSize = 7

ax = axes[0]
iters_pre = [0, 5, 10, 15, 20, 25, 30, 35, 39]
iters_post = [40, 42, 44, 46, 48, 50, 52, 54]
iteraA = iters_pre + iters_post
ax.plot(iteraA[1:], p_deltaNorms, label=r"$\Delta_{t,j}(\theta)$", color="C0", linestyle="-", marker="o", markersize=markSize, linewidth=linewidth)
ax.plot(iteraA[1:], q_deltaNorms, label=r"$\Delta_{t,j}(\phi)$", color="C1", linestyle="--", marker="s", markersize=markSize, linewidth=linewidth)
ax.set_xlabel("Epochs", size=labelSize)
ax.tick_params(axis="both", labelsize=labelSize)
ax.set_ylabel(r"$\Delta_{t,j}(\varphi)$")
ax.xaxis.set_label_coords(0.5, -0.15)

ax2= ax.twinx()
ax2.plot(iteraA[1:], p_rewards[1:], label=r"$J_p(\theta_t)$", color="C3", linestyle="--", marker="x", markersize=markSize, linewidth=linewidth)
ax2.plot(iteraA[1:], q_rewards[1:], label=r"$J_q(\phi_t)$", color="C4", linestyle="--", marker="d", markersize=markSize, linewidth=linewidth)
ax2.set_ylabel(r"$J_p(\theta_t)/J_q(\phi_t)$", size=labelSize)
ax2.set_ylabel(ax2.get_ylabel(), labelpad=2)
ax2.yaxis.set_label_coords(1.12, 0.5)
ax2.tick_params(axis="y", pad=2, labelsize=labelSize)
plt.grid(True, alpha=0.3)
plt.tight_layout()

axes[1].plot(["A1", "A2", "A3", "A4"], p_rewards_final[0:7:2], label=r"$J_p(\theta^*),\ \lambda_{\mathcal{H}}^{\theta}=0.5$", 
             color="C0", linestyle="-", marker="o", markersize=markSize, linewidth=linewidth)
axes[1].plot(["A1", "A2", "A3", "A4"], p_rewards_final[1:8:2], label=r"$J_p(\theta^*),\ \lambda_{\mathcal{H}}^{\theta}=0.6$", 
             color="red", linestyle="--", marker="x", markersize=markSize, linewidth=linewidth)
axes[1].axhline(y=p_final_reward_A5_cu5, color="C0", linestyle=":", linewidth=linewidth-0.5)
axes[1].axhline(y=p_final_reward_A5_cu6, color="red", linestyle=":", linewidth=linewidth-0.5)
axes[1].axhline(y=p_final_reward_A6_cu5, color="C0", linestyle="-.", linewidth=linewidth-0.5)
axes[1].axhline(y=p_final_reward_A6_cu6, color="red", linestyle="-.", linewidth=linewidth-0.5)
axes[1].tick_params(axis="both", labelsize=labelSize)
axes[1].grid(True, alpha=0.3)

axes[2].plot(["A1", "A2", "A3", "A4"], q_rewards_final[0:7:2], label=r"$J_q(\phi^*),\ \lambda_{\mathcal{H}}^{\theta}=0.5$", 
             color="C0", linestyle="-", marker="o", markersize=markSize, linewidth=linewidth)
axes[2].plot(["A1", "A2", "A3", "A4"], q_rewards_final[1:8:2], label=r"$J_q(\phi^*),\ \lambda_{\mathcal{H}}^{\theta}=0.6$", 
             color="red", linestyle="--", marker="x", markersize=markSize, linewidth=linewidth)
axes[2].axhline(y=q_final_reward_A5_cu5, color="C0", linestyle=":", linewidth=linewidth-0.5)
axes[2].axhline(y=q_final_reward_A5_cu6, color="red", linestyle=":", linewidth=linewidth-0.5)
axes[2].axhline(y=q_final_reward_A6_cu5, color="C0", linestyle="-.", linewidth=linewidth-0.5)
axes[2].axhline(y=q_final_reward_A6_cu6, color="red", linestyle="-.", linewidth=linewidth-0.5)
axes[2].grid(True, alpha=0.3)
axes[2].tick_params(axis="both", labelsize=labelSize)

for ax in axes:
    ax.set_ylabel(ax.get_ylabel(), labelpad=2)
    ax.yaxis.set_label_coords(-0.12, 0.5)
    ax.tick_params(axis="y", pad=2) 

fig.savefig(f"figures/curve_converg_A{setting}_cu{cu5or6}_Jp_Jq_A1_6.png", dpi=800, bbox_inches="tight")   
fig.savefig(f"figures/curve_converg_A{setting}_cu{cu5or6}_Jp_Jq_A1_6.svg", dpi=1800, format="svg")    

# all rewards values during iterations

In [ ]:
import os, shutil, subprocess,sys
import torch
import json
import numpy as np
import matplotlib as mpl

PDFTEXBIN = '/opt/local/texlive/texlive-2018/2018/bin/x86_64-linux/pdflatex'
TEXBIN = os.path.dirname(PDFTEXBIN)

os.environ['PATH'] = TEXBIN + os.pathsep + os.environ.get('PATH', '')
mpl.use("pgf")
mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "font.family": "serif",
    "pgf.preamble": r"""
\usepackage[T1]{fontenc}
\usepackage{newtxtext,newtxmath}
""",
    "font.size": 8,          
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "lines.linewidth": 2.0,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

TEXTWIDTH_IN = 6.75 * 2

fig = plt.figure(figsize=(TEXTWIDTH_IN, 2 * 2))
spacer_ratio=0.01
gs = fig.add_gridspec(3, 5, width_ratios=[1, 1, spacer_ratio, 1, 1], wspace=0.25, hspace=0.4)
axes = [
    [fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[0, 3]),
    fig.add_subplot(gs[0, 4]),], 
    [
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[1, 3]),     
    fig.add_subplot(gs[1, 4]),],
    [
    fig.add_subplot(gs[2, 0]),
    fig.add_subplot(gs[2, 1]),
    fig.add_subplot(gs[2, 3]),
    fig.add_subplot(gs[2, 4]),],
]

ax_spacer_1 = fig.add_subplot(gs[0, 2])
ax_spacer_1.axis("off")
ax_spacer_2 = fig.add_subplot(gs[1, 2])
ax_spacer_2.axis("off")
ax_spacer_3 = fig.add_subplot(gs[2, 2])
ax_spacer_3.axis("off")

markSize = 1
linewidth = 1.2
labelSize = 9
colors = ["C0", "C2", "C1", "C3"]
markers = ['None', 'None']

axes[0][0].plot(p_rewards_A1_cu5, color=colors[0], linestyle="-", marker=markers[0], markersize=markSize, 
             linewidth=linewidth, label=r"$J_p(\theta_t),\ \lambda_{\mathcal{H}}^{\theta}=0.5$")
axes[0][0].plot(p_rewards_A1_cu6, color=colors[1], linestyle="--", marker=markers[1], markersize=markSize, 
             linewidth=linewidth, label=r"$J_p(\theta_t),\ \lambda_{\mathcal{H}}^{\theta}=0.6$")
axes[0][0].tick_params(axis="both", labelsize=labelSize)

axes[0][1].plot(q_rewards_A1_cu5, color=colors[2], linestyle="-", marker=markers[0], markersize=markSize, 
             linewidth=linewidth, label=r"$J_q(\phi_t),\ \lambda_{\mathcal{H}}^{\theta}=0.5$")
axes[0][1].plot(q_rewards_A1_cu6, color=colors[3], linestyle="--", marker=markers[1], markersize=markSize, 
             linewidth=linewidth, label=r"$J_q(\phi_t),\ \lambda_{\mathcal{H}}^{\theta}=0.6$")
axes[0][1].tick_params(axis="both", labelsize=labelSize)

axes[0][2].plot(p_rewards_A2_cu5, color=colors[0], linestyle="-", marker=markers[0], markersize=markSize, 
             linewidth=linewidth, label=r"$J_p(\theta_t),\ \lambda_{\mathcal{H}}^{\theta}=0.5$")
axes[0][2].plot(p_rewards_A2_cu6, color=colors[1], linestyle="--", marker=markers[1], markersize=markSize, 
             linewidth=linewidth, label=r"$J_p(\theta_t),\ \lambda_{\mathcal{H}}^{\theta}=0.6$")
axes[0][2].tick_params(axis="both", labelsize=labelSize)

axes[0][3].plot(q_rewards_A2_cu5, color=colors[2], linestyle="-", marker=markers[0], markersize=markSize, 
             linewidth=linewidth, label=r"$J_q(\phi_t),\ \lambda_{\mathcal{H}}^{\theta}=0.5$")
axes[0][3].plot(q_rewards_A2_cu6, color=colors[3], linestyle="--", marker=markers[1], markersize=markSize, 
             linewidth=linewidth, label=r"$J_q(\phi_t),\ \lambda_{\mathcal{H}}^{\theta}=0.6$")
axes[0][3].tick_params(axis="both", labelsize=labelSize)

axes[1][0].plot(p_rewards_A3_cu5, color=colors[0], linestyle="-", marker=markers[0], markersize=markSize, 
             linewidth=linewidth, label=r"$J_p(\theta_t),\ \lambda_{\mathcal{H}}^{\theta}=0.5$")
axes[1][0].plot(p_rewards_A3_cu6, color=colors[1], linestyle="--", marker=markers[1], markersize=markSize, 
             linewidth=linewidth, label=r"$J_p(\theta_t),\ \lambda_{\mathcal{H}}^{\theta}=0.6$")
axes[1][0].tick_params(axis="both", labelsize=labelSize)
axes[1][1].plot(q_rewards_A3_cu5, color=colors[2], linestyle="-", marker=markers[0], markersize=markSize,
             linewidth=linewidth, label=r"$J_q(\phi_t),\ \lambda_{\mathcal{H}}^{\theta}=0.5$")
axes[1][1].plot(q_rewards_A3_cu6, color=colors[3], linestyle="--", marker=markers[1], markersize=markSize, 
             linewidth=linewidth, label=r"$J_q(\phi_t),\ \lambda_{\mathcal{H}}^{\theta}=0.6$")
axes[1][1].tick_params(axis="both", labelsize=labelSize)

axes[1][2].plot(p_rewards_A4_cu5, color=colors[0], linestyle="-", marker=markers[0], markersize=markSize, 
             linewidth=linewidth, label=r"$J_p(\theta_t),\ \lambda_{\mathcal{H}}^{\theta}=0.5$")
axes[1][2].plot(p_rewards_A4_cu6, color=colors[1], linestyle="--", marker=markers[1], markersize=markSize, 
             linewidth=linewidth, label=r"$J_p(\theta_t),\ \lambda_{\mathcal{H}}^{\theta}=0.6$")
axes[1][2].tick_params(axis="both", labelsize=labelSize)
axes[1][3].plot(q_rewards_A4_cu5, color=colors[2], linestyle="-", marker=markers[0], markersize=markSize, 
             linewidth=linewidth, label=r"$J_q(\phi_t),\ \lambda_{\mathcal{H}}^{\theta}=0.5$")
axes[1][3].plot(q_rewards_A4_cu6, color=colors[3], linestyle="--", marker=markers[1], markersize=markSize, 
             linewidth=linewidth, label=r"$J_q(\phi_t),\ \lambda_{\mathcal{H}}^{\theta}=0.6$")
axes[1][3].tick_params(axis="both", labelsize=labelSize)

axes[2][0].plot(p_rewards_A5_cu5, color=colors[0], linestyle="-", marker=markers[0], markersize=markSize, 
             linewidth=linewidth, label=r"$J_p(\theta_t),\ \lambda_{\mathcal{H}}^{\theta}=0.5$")
axes[2][0].plot(p_rewards_A5_cu6, color=colors[1], linestyle="--", marker=markers[1], markersize=markSize, 
             linewidth=linewidth, label=r"$J_p(\theta_t),\ \lambda_{\mathcal{H}}^{\theta}=0.6$")
axes[2][0].tick_params(axis="both", labelsize=labelSize)
axes[2][1].plot(q_rewards_A5_cu5, color=colors[2], linestyle="-", marker=markers[0], markersize=markSize, 
             linewidth=linewidth, label=r"$J_q(\phi_t),\ \lambda_{\mathcal{H}}^{\theta}=0.5$")
axes[2][1].plot(q_rewards_A5_cu6, color=colors[3], linestyle="--", marker=markers[1], markersize=markSize, 
             linewidth=linewidth, label=r"$J_q(\phi_t),\ \lambda_{\mathcal{H}}^{\theta}=0.6$")
axes[2][1].tick_params(axis="both", labelsize=labelSize)

axes[2][2].plot(p_rewards_A6_cu5, color=colors[0], linestyle="-", marker=markers[0], markersize=markSize, 
             linewidth=linewidth, label=r"$J_p(\theta_t),\ \lambda_{\mathcal{H}}^{\theta}=0.5$")
axes[2][2].plot(p_rewards_A6_cu6, color=colors[1], linestyle="--", marker=markers[1], markersize=markSize, 
             linewidth=linewidth, label=r"$J_p(\theta_t),\ \lambda_{\mathcal{H}}^{\theta}=0.6$")
axes[2][2].tick_params(axis="both", labelsize=labelSize)
axes[2][3].plot(q_rewards_A6_cu5, color=colors[2], linestyle="-", marker=markers[0], markersize=markSize, 
             linewidth=linewidth, label=r"$J_q(\phi_t),\ \lambda_{\mathcal{H}}^{\theta}=0.5$")
axes[2][3].plot(q_rewards_A6_cu6, color=colors[3], linestyle="--", marker=markers[1], markersize=markSize, 
             linewidth=linewidth, label=r"$J_q(\phi_t),\ \lambda_{\mathcal{H}}^{\theta}=0.6$")
axes[2][3].tick_params(axis="both", labelsize=labelSize)

for ax_ in axes:
    for ax in ax_:
        ax.set_ylabel(ax.get_ylabel(), labelpad=2)
        ax.yaxis.set_label_coords(-0.12, 0.5)
        ax.tick_params(axis="y", pad=2) 
        ax.grid(True, alpha=0.3)
        
plt.grid(True, alpha=0.3)
plt.tight_layout()

fig.savefig(f"figures/curve_allrewards.png", dpi=800, bbox_inches="tight")     
fig.savefig(f"figures/curve_allrewards.svg", dpi=1800, format="svg")     